In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import os
from tqdm.auto import tqdm
import re
from pathlib import Path
import glob

In [2]:
path_to_plot_data = "//home/gzu5140/Keerthana_b1042/grnInference/plot_data/"
path_to_plots = "/home/gzu5140/Keerthana_b1042/grnInference/plots/"
path_to_code_repo = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/"

## Effect of varying Hill constant

In [2]:
def extract_k_values(simulation_folder):
    """
    Extract the value of Hill constant values from filenames in the simulation folder.
    
    Args:
        simulation_folder (str): Path to the folder containing simulation files
    
    Returns:
        list: List of tuples containing (filename, k_value)
    """
    
    # Check if folder exists
    if not os.path.exists(simulation_folder):
        print(f"Error: Folder {simulation_folder} does not exist")
        return []
    
    # Pattern to match scale_k_ followed by a number (including decimal)
    k_pattern = r'scale_k_([0-9]*\.?[0-9]+)'
    
    k_values = []
    
    # Iterate through all files in the folder
    for filename in os.listdir(simulation_folder):
        # Only process CSV files (or remove this filter if you want all files)
        if filename.endswith('.csv') and filename.startswith('df') and "rep" in filename:
            # Search for the k value pattern in the filename
            match = re.search(k_pattern, filename)
            if match:
                k_value = float(match.group(1))  # Extract and convert to float
                k_values.append((filename, k_value))
            else:
                print(f"Warning: No k value found in filename: {filename}")
    
    return k_values

def print_k_values(k_values):
    """Print the extracted k values in a formatted way"""
    if not k_values:
        print("No k values found.")
        return
    
    print(f"Found {len(k_values)} files with k values:")
    print("-" * 80)
    print(f"{'Filename':<60} {'K Value':<10}")
    print("-" * 80)
    
    # Sort by k value for better readability
    k_values_sorted = sorted(k_values, key=lambda x: x[1])
    
    for filename, k_value in k_values_sorted:
        print(f"{filename:<60} {k_value:<10}")

def get_unique_k_values(k_values):
    """Get unique k values"""
    unique_k = sorted(list(set([k for _, k in k_values])))
    return unique_k


In [3]:
def calculate_gene_correlation_simulation(df_path, k, t1=1, t2=20, gene_list=['gene_1', 'gene_2']): 
    '''
    Calculate gene correlation for different files 
    '''
    sim_data = {} 
    df = pd.read_csv(df_path) 
    expression_t1 = df[df['time_step'] == t1] 
    expression_t2 = df[df['time_step'] == t2] 
    sim_data["k"] = k 
    for gene in gene_list: 
        # Mean 
        sim_data[f"mean_{gene}_t1"] = expression_t1[f"{gene}_mRNA"].mean() 
        sim_data[f"mean_{gene}_t2"] = expression_t2[f"{gene}_mRNA"].mean() 
        # Variance 
        sim_data[f"var_{gene}_t1"] = expression_t1[f"{gene}_mRNA"].var() 
        sim_data[f"var_{gene}_t2"] = expression_t2[f"{gene}_mRNA"].var() 
        
        # Spearman correlations 
    sim_data["corr_t1"], _ = spearmanr( expression_t1[f"{gene_list[0]}_mRNA"], expression_t1[f"{gene_list[1]}_mRNA"] ) 
    sim_data["corr_t2"], _ = spearmanr( expression_t2[f"{gene_list[0]}_mRNA"], expression_t2[f"{gene_list[1]}_mRNA"] ) 
    return sim_data

In [88]:
# Simulation folder containing hill constant simulations
simulation_folder = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/hill_constant_effect/simulations/"
k_values = extract_k_values(simulation_folder)

# Optional: Create a dictionary mapping k values to filenames
k_to_files = {}
for filename, k_value in k_values:
    if k_value not in k_to_files:
        k_to_files[k_value] = []
    k_to_files[k_value].append(filename)

In [90]:
data = []
for k, paths in k_to_files.items():
    for path in paths:   # loop over ALL replicates
        full_path = f"{simulation_folder}/{path}"
        sim_data = calculate_gene_correlation_simulation(full_path, k)
        sim_data["rep_file"] = path   # optional: keep track of replicate filename
        data.append(sim_data)

results = pd.DataFrame(data)
results.sort_values("k", axis=0, ascending=True, inplace=True)
results = results.reset_index(drop=True)
results.to_csv(f"{path_to_plot_data}/effect_K.csv")

### Asymmetric splitting

In [9]:
def extract_bias_values(simulation_folder):
    """
    Extract the value of Hill constant values from filenames in the simulation folder.
    
    Args:
        simulation_folder (str): Path to the folder containing simulation files
    
    Returns:
        list: List of tuples containing (filename, k_value)
    """
    
    # Check if folder exists
    if not os.path.exists(simulation_folder):
        print(f"Error: Folder {simulation_folder} does not exist")
        return []
    
    # Pattern to match scale_k_ followed by a number (including decimal)
    k_pattern = r'bias_([0-9]*\.?[0-9]+)'
    
    k_values = []
    
    # Iterate through all files in the folder
    for filename in os.listdir(simulation_folder):
        # Only process CSV files (or remove this filter if you want all files)
        if filename.endswith('.csv') and filename.startswith('df'):
            # Search for the k value pattern in the filename
            match = re.search(k_pattern, filename)
            if match:
                k_value = float(match.group(1))  # Extract and convert to float
                k_values.append((filename, k_value))
            else:
                print(f"Warning: No k value found in filename: {filename}")
    
    return k_values

def print_k_values(k_values):
    """Print the extracted k values in a formatted way"""
    if not k_values:
        print("No k values found.")
        return
    
    print(f"Found {len(k_values)} files with k values:")
    print("-" * 80)
    print(f"{'Filename':<60} {'K Value':<10}")
    print("-" * 80)
    
    # Sort by k value for better readability
    k_values_sorted = sorted(k_values, key=lambda x: x[1])
    
    for filename, k_value in k_values_sorted:
        print(f"{filename:<60} {k_value:<10}")

def get_unique_k_values(k_values):
    """Get unique k values"""
    unique_k = sorted(list(set([k for _, k in k_values])))
    return unique_k


In [4]:
def calculate_gene_correlation_simulation(df_path, k, t1=1, t2=20, gene_list=['gene_1', 'gene_2']): 
    '''
    Calculate gene correlation for different files 
    '''
    sim_data = {} 
    df = pd.read_csv(df_path) 
    expression_t1 = df[df['time_step'] == t1] 
    expression_t2 = df[df['time_step'] == t2] 
    sim_data["k"] = k 
    for gene in gene_list: 
        # Mean 
        sim_data[f"mean_{gene}_t1"] = expression_t1[f"{gene}_mRNA"].mean() 
        sim_data[f"mean_{gene}_t2"] = expression_t2[f"{gene}_mRNA"].mean() 
        # Variance 
        sim_data[f"var_{gene}_t1"] = expression_t1[f"{gene}_mRNA"].var() 
        sim_data[f"var_{gene}_t2"] = expression_t2[f"{gene}_mRNA"].var() 
        
        # Spearman correlations 
    sim_data["corr_t1"], _ = spearmanr( expression_t1[f"{gene_list[0]}_mRNA"], expression_t1[f"{gene_list[1]}_mRNA"] ) 
    sim_data["corr_t2"], _ = spearmanr( expression_t2[f"{gene_list[0]}_mRNA"], expression_t2[f"{gene_list[1]}_mRNA"] ) 
    return sim_data

In [ ]:

def get_cross_correlations(rep_0_t1,
                                   rep_1_t2,
                                   gene_pairs,
                                   type_comparison="twin"):
    """
    Computes directional Spearman correlations between gene_1 (at t1) and gene_2 (at t2),
    and returns both raw and normalized directional matrices.

    Parameters
    ----------
    rep0_t1 : pd.DataFrame
        Simulation data at time t1 with one twin, with columns: 'replicate', 'clone_id', '{gene}_mRNA'.

    rep1_t2 : pd.DataFrame
        Simulation data at time t2 with the other twin, same structure as t1.

    gene_pairs : list of tuple
        List of (gene_1, gene_2) pairs to analyze directionally.

    type_comparison : str, optional
        If "twin", checks that clone_ids are aligned. If "random", no check is performed.

    Returns
    -------
    raw_matrix : pd.DataFrame
        Raw correlation matrix (gene_1 at t1 → gene_2 at t2).

    normalized_matrix : pd.DataFrame
        Normalized correlation matrix.
    """
    gene_list = sorted(set(g for pair in gene_pairs for g in pair))

    # Separate replicates for t1 and t2
    rep_0_t1 = rep_0_t1.sort_values("clone_id").reset_index(drop=True)
    rep_1_t2 = rep_1_t2.sort_values("clone_id").reset_index(drop=True)


    all_genes = list(set(gene_1 for gene_1, _ in gene_pairs) | set(gene_2 for _, gene_2 in gene_pairs))
    self_pairs = [(gene, gene) for gene in all_genes if (gene, gene) not in gene_pairs]
    gene_pairs += self_pairs

    if type_comparison == "twin":
        if not rep_0_t1["clone_id"].equals(rep_1_t2["clone_id"]):
                print("Clone IDs do not match between replicates:")
                mismatched_ids = rep_1_t2[~rep_0_t1["clone_id"].isin(rep_0_t1["clone_id"])]
                print(mismatched_ids["clone_id"].unique())
                raise ValueError(f"Mismatch in clone_id")

    # Compute raw directional correlations
    raw_matrix = pd.DataFrame(index=gene_list, columns=gene_list, dtype=float)
    for gene_1, gene_2 in gene_pairs:
        x = rep_0_t1[f"{gene_1}_mRNA"]
        y = rep_1_t2[f"{gene_2}_mRNA"]
        corr = spearmanr(x, y).correlation
        raw_matrix.loc[gene_1, gene_2] = corr
    return raw_matrix

In [10]:
# Simulation folder containing hill constant simulations
simulation_folder = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/binomial_partition_check/A_to_B/"
k_values = extract_bias_values(simulation_folder)

# Optional: Create a dictionary mapping k values to filenames
k_to_files = {}
for filename, k_value in k_values:
    if k_value not in k_to_files:
        k_to_files[k_value] = []
    k_to_files[k_value].append(filename)

In [16]:
from tqdm.notebook import tqdm
data = []
for k, paths in tqdm(k_to_files.items()):
    for path in paths:   # loop over ALL replicates
        full_path = f"{simulation_folder}/{path}"
        sim_data = calculate_gene_correlation_simulation(full_path, k)
        sim_data["rep_file"] = path   # optional: keep track of replicate filename
        data.append(sim_data)

results = pd.DataFrame(data)
results.sort_values("k", axis=0, ascending=True, inplace=True)
# results = results.reset_index(drop=True)
# results.to_csv(f"{path_to_plot_data}/effect_symmetry.csv")

  0%|          | 0/12 [00:00<?, ?it/s]

## Effect of $k_{add}$

In [ ]:
from joblib import Parallel, delayed
import pandas as pd
import os, glob, re
from tqdm.auto import tqdm

path_to_simulation_files_for_k_add_activation = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/k_add_simulations/A_to_B_reps/"
path_to_k_add_parameter_csv = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/k_add_simulations/simulation_details/effect_of_k_add_sampling_positive_with_reps.csv"

# Load parameter details with explicit param_row
details_df = pd.read_csv(path_to_k_add_parameter_csv).reset_index().rename(columns={"index": "param_row"})

pattern = re.compile(r"df_row_(\d+)_(\d+)_")

def process_file(filepath, details_df):
    fname = os.path.basename(filepath)
    if fname.startswith("sim"):
        return None

    # Extract param_row
    m = pattern.search(fname)
    if not m:
        return None
    _, param_row = int(m.group(1)), int(m.group(2))

    # Lookup k_add
    row = details_df[details_df["param_row"] == param_row]
    if row.empty:
        return None
    r = row.iloc[0]
    k = r["k_add_gene_1_to_gene_2"]
    k_off = r["k_off"]

    # Run analysis (assuming this returns a dict or DataFrame with correlation values)
    sim_data = process_simulation(filepath, k)
    sim_data['k_add'] = k
    sim_data['k_off'] = k_off

    return  sim_data

# Get all files
all_files = glob.glob(os.path.join(path_to_simulation_files_for_k_add_activation, "*.csv"))

# Parallel loop
results = Parallel(n_jobs=-8)(  # use all available cores
    delayed(process_file)(filepath, details_df) for filepath in tqdm(all_files)
)

# Filter out None results
results = [r for r in results if r is not None]

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.sort_values("k_add", axis=0, ascending=True, inplace=True)
results_df.reset_index(drop=True, inplace=True)

results_df.to_csv(f"{path_to_plot_data}/k_add_activation.csv")


In [ ]:
def process_simulation(df_path, k, t1=1, t2=20, gene_list=['gene_1', 'gene_2']): 
    sim_data = {} 
    df = pd.read_csv(df_path) 
    expression_t1 = df[df['time_step'] == t1] 
    expression_t2 = df[df['time_step'] == t2] 
    sim_data["k"] = k 
    for gene in gene_list: 
        # Mean 
        sim_data[f"mean_{gene}_t1"] = expression_t1[f"{gene}_mRNA"].mean() 
        sim_data[f"mean_{gene}_t2"] = expression_t2[f"{gene}_mRNA"].mean() 
        # Variance 
        sim_data[f"var_{gene}_t1"] = expression_t1[f"{gene}_mRNA"].var() 
        sim_data[f"var_{gene}_t2"] = expression_t2[f"{gene}_mRNA"].var() 
        
        # Spearman correlations 
    sim_data["corr_t1"], _ = spearmanr( expression_t1[f"{gene_list[0]}_mRNA"], expression_t1[f"{gene_list[1]}_mRNA"] ) 
    sim_data["corr_t2"], _ = spearmanr( expression_t2[f"{gene_list[0]}_mRNA"], expression_t2[f"{gene_list[1]}_mRNA"] ) 
    return sim_data

In [ ]:
# ---- Main loop ----
path_to_simulation_files_for_k_add_repression = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/k_add_simulations/A_rep_B_with_reps/"
path_to_k_add_parameter_csv_repression = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/k_add_simulations/simulation_details/effect_of_k_add_sampling_repression_with_reps.csv"

# Load parameter details with explicit param_row
details_df = pd.read_csv(path_to_k_add_parameter_csv_repression).reset_index().rename(columns={"index": "param_row"})

from joblib import Parallel, delayed
import pandas as pd
import os, glob, re
from tqdm.auto import tqdm

pattern = re.compile(r"df_row_(\d+)_(\d+)_")

def process_file(filepath, details_df):
    fname = os.path.basename(filepath)
    if fname.startswith("sim"):
        return None

    # Extract param_row
    m = pattern.search(fname)
    if not m:
        return None
    _, param_row = int(m.group(1)), int(m.group(2))

    # Lookup k_add
    row = details_df[details_df["param_row"] == param_row]
    if row.empty:
        return None
    r = row.iloc[0]
    k = r["k_add_gene_1_to_gene_2"]
    k_off = r["k_off"]

    # Run analysis (assuming this returns a dict or DataFrame with correlation values)
    sim_data = process_simulation(filepath, k)
    sim_data['k_add'] = k
    sim_data['k_off'] = k_off

    return  sim_data

# Get all files
print(sim_folder)
all_files = glob.glob(os.path.join(path_to_simulation_files_for_k_add_repression, "df*.csv"))

# Parallel loop
results = Parallel(n_jobs=8)(  # use all available cores
    delayed(process_file)(filepath, details_df) for filepath in tqdm(all_files)
)

# Filter out None results
results = [r for r in results if r is not None]

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.sort_values("k_add", axis=0, ascending=True, inplace=True)
results_df.reset_index(drop=True, inplace=True)
results_df.to_csv(f"{path_to_plot_data}/k_add_repression.csv")

## Twin correlation over time

In [3]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import importlib
import matplotlib.pyplot as plt
from numba import set_num_threads, get_num_threads

sys.path.append(str(path_to_code_repo))

from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)

from TwINFER_function_scripts.correlation_analysis_functions import (
    
    calculate_pairwise_gene_gene_correlation_matrix,
    check_system_in_steady_state,
    check_gene_gene_correlation_threshold,
    calculate_twin_random_pair_correlations,
    differentiate_single_state_reg_and_multiple_states,
    get_cross_correlations,
    identify_actual_directed_edges
)

from TwINFER_function_scripts.correlation_analysis_helpers import (
    extract_param_index,
    read_input_matrix,
    get_param_data, 
    plot_matrix_as_heatmap,
    print_summary,
    plot_network
)

In [4]:
def split_and_merge_simulations(path_to_simulation_files):
    simulation_1 = pd.read_csv(path_to_simulation_files[0])
    simulation_2 = pd.read_csv(path_to_simulation_files[1])
    clone_ids = sorted(simulation_1['clone_id'].unique())
    half_point = len(clone_ids) // 2
    clones_from_sim1 = clone_ids[:half_point]
    clones_from_sim2 = clone_ids[half_point:]
    sim1_subset = simulation_1[simulation_1['clone_id'].isin(clones_from_sim1)]
    sim2_subset = simulation_2[simulation_2['clone_id'].isin(clones_from_sim2)]
    return pd.concat([sim1_subset, sim2_subset], ignore_index=True)

### 2 states correlation over time - both twin and random

In [ ]:
low_csvs

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr
t1 = 1
t2 = 20
# Paths - simulation from figure 2
folder_name = "A_B_2_states"
low_folder  = Path("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/binomial_partition_check/A_B_low")
high_folder = Path("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/binomial_partition_check/A_B_high")

low_csvs  = sorted(low_folder.glob("df_rows_*bias_0.5*.csv"))
high_csvs = sorted(high_folder.glob("df_rows_*bias_0.5*.csv"))

path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_B.txt"
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]
results = []
# Parameters
seed = 1010

rng = np.random.default_rng(seed)

# --- match by replicate number ---
import re

def get_rep_number(path):
    """
    Extract replicate number appearing after 'bias_<value>_'.
    Example:
    ..._bias_0.5_0_06dc79ec.csv  -> 0
    """
    m = re.search(r"bias_[0-9.]+_(\d+)", path.stem)
    if m is None:
        raise ValueError(f"Could not extract replicate number from {path.name}")
    return int(m.group(1))

# dictionary: rep → path
low_by_rep  = {get_rep_number(p): p for p in low_csvs}
high_by_rep = {get_rep_number(p): p for p in high_csvs}

# process only reps present in BOTH
common_reps = sorted(set(low_by_rep) & set(high_by_rep))

for rep in common_reps:
    pair_paths = [low_by_rep[rep], high_by_rep[rep]]
    
    # --- 1. split + merge ---
    simulation = split_and_merge_simulations(pair_paths)
    n_clones_simulation = simulation['clone_id'].nunique()
    
    # Shuffle clones
    clone_ids_shuffled = np.random.permutation(n_clones_simulation)
    n1 = n2 = n_clones_simulation // 4
    t1_clones = clone_ids_shuffled[:n1]
    t2_clones = clone_ids_shuffled[n1:n1 + n2]
    across_t_clones = clone_ids_shuffled[n1 + n2:]
    
    t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]
    t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]

    # Across_t: pick exactly one random twin per clone_id
    # One cell per clone at t1
    across_t_twin1 = (
        simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
    )
    
    across_t_twin2 = (
        simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
    )

    # Reset index for cleanliness
    t1_twins = t1_twins.reset_index(drop=True)
    t2_twins = t2_twins.reset_index(drop=True)
    across_t_twin1 = across_t_twin1.reset_index(drop=True)
    across_t_twin2 = across_t_twin2.reset_index(drop=True)

    all_t1_t2_measurements = pd.concat(
    [t1_twins, t2_twins, across_t_twin1, across_t_twin2],
    ignore_index=True
    )

    # Loop across all time points available
    for t1 in sorted(simulation['time_step'].unique()):
        # Twins at this time
        t1_twins = simulation[(simulation['clone_id'].isin(t1_clones)) & (simulation['time_step'] == t1)]
        t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]

        # Across_t: pick exactly one random twin per clone_id
        # One cell per clone at t1
        across_t_twin1 = (
            simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
        )
        
        across_t_twin2 = (
            simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
        )

        # Reset index for cleanliness
        t1_twins = t1_twins.reset_index(drop=True)
        t2_twins = t2_twins.reset_index(drop=True)
        across_t_twin1 = across_t_twin1.reset_index(drop=True)
        across_t_twin2 = across_t_twin2.reset_index(drop=True)

        all_t1_t2_measurements = pd.concat(
        [t1_twins, t2_twins, across_t_twin1, across_t_twin2],
        ignore_index=True
        )
        if t1_twins.empty:
            continue
        
        # Pairwise gene–gene correlation matrix
        twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t2 = calculate_twin_random_pair_correlations(
        all_t1_t2_measurements, t1_twins, gene_list
        )
        gene_corr = spearmanr(all_t1_t2_measurements['gene_1_mRNA'], all_t1_t2_measurements['gene_2_mRNA'])
        # Twin correlation (example: gene_1 vs gene_2)
        twin_corr = twin_pair_correlation_matrix_t1.loc['gene_1', 'gene_2']
        random_corr = random_pair_correlation_matrix_t2.loc['gene_1', 'gene_2']
        results.append({
            "file_low": low_by_rep[rep].name,
            "file_high": high_by_rep[rep].name,
            "time_step": t1,
            "gene_corr":gene_corr,
            "twin_corr_gene1_gene2": twin_corr,
            "random_corr_gene1_gene2": random_corr
        })

# Collect into DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["file_low", "time_step"]).reset_index(drop=True)

# # Save table (optional)
results_df.to_csv(f"{path_to_plot_data}/correlation_over_time_binomial_{folder_name}.csv", index=False)


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
t1 = 1
t2 = 20
# Paths to simulation in figure 2
folder_name = "A_to_B_2_states"
low_folder  = Path("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/binomial_partition_check/A_to_B_low")
high_folder = Path("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/binomial_partition_check/A_to_B_high")

low_csvs  = sorted(low_folder.glob("df_rows_*bias_0.5_*.csv"))
high_csvs = sorted(high_folder.glob("df_rows_*bias_0.5_*.csv"))

path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_to_B.txt"
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]
results = []
# Parameters
seed = 1010


rng = np.random.default_rng(seed)

# --- match by replicate number ---
def get_rep_number(path):
    """
    Extract replicate number appearing after 'bias_<value>_'.
    Example:
    ..._bias_0.5_0_06dc79ec.csv  -> 0
    """
    m = re.search(r"bias_[0-9.]+_(\d+)", path.stem)
    if m is None:
        raise ValueError(f"Could not extract replicate number from {path.name}")
    return int(m.group(1))

# dictionary: rep → path
low_by_rep  = {get_rep_number(p): p for p in low_csvs}
high_by_rep = {get_rep_number(p): p for p in high_csvs}

# process only reps present in BOTH
common_reps = sorted(set(low_by_rep) & set(high_by_rep))

for rep in common_reps:
    pair_paths = [low_by_rep[rep], high_by_rep[rep]]
    
    # --- 1. split + merge ---
    simulation = split_and_merge_simulations(pair_paths)
    n_clones_simulation = simulation['clone_id'].nunique()
    
    # Shuffle clones
    clone_ids_shuffled = np.random.permutation(n_clones_simulation)
    n1 = n2 = n_clones_simulation // 4
    t1_clones = clone_ids_shuffled[:n1]
    t2_clones = clone_ids_shuffled[n1:n1 + n2]
    across_t_clones = clone_ids_shuffled[n1 + n2:]
    
    t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]
    t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]

    # Across_t: pick exactly one random twin per clone_id
    # One cell per clone at t1
    across_t_twin1 = (
        simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
    )
    
    across_t_twin2 = (
        simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
    )

    # Reset index for cleanliness
    t1_twins = t1_twins.reset_index(drop=True)
    t2_twins = t2_twins.reset_index(drop=True)
    across_t_twin1 = across_t_twin1.reset_index(drop=True)
    across_t_twin2 = across_t_twin2.reset_index(drop=True)

    all_t1_t2_measurements = pd.concat(
    [t1_twins, t2_twins, across_t_twin1, across_t_twin2],
    ignore_index=True
    )

    # Loop across all time points available
    for t1 in sorted(simulation['time_step'].unique()):
        # Twins at this time
        t1_twins = simulation[(simulation['clone_id'].isin(t1_clones)) & (simulation['time_step'] == t1)]
        t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]

        # Across_t: pick exactly one random twin per clone_id
        # One cell per clone at t1
        across_t_twin1 = (
            simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
        )
        
        across_t_twin2 = (
            simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
        )

        # Reset index for cleanliness
        t1_twins = t1_twins.reset_index(drop=True)
        t2_twins = t2_twins.reset_index(drop=True)
        across_t_twin1 = across_t_twin1.reset_index(drop=True)
        across_t_twin2 = across_t_twin2.reset_index(drop=True)

        all_t1_t2_measurements = pd.concat(
        [t1_twins, t2_twins, across_t_twin1, across_t_twin2],
        ignore_index=True
        )
        if t1_twins.empty:
            continue
        
        # Pairwise gene–gene correlation matrix
        twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t2 = calculate_twin_random_pair_correlations(
        all_t1_t2_measurements, t1_twins, gene_list
        )
        gene_corr = spearmanr(all_t1_t2_measurements['gene_1_mRNA'], all_t1_t2_measurements['gene_2_mRNA'])
        # Twin correlation (example: gene_1 vs gene_2)
        twin_corr = twin_pair_correlation_matrix_t1.loc['gene_1', 'gene_2']
        random_corr = random_pair_correlation_matrix_t2.loc['gene_1', 'gene_2']
        results.append({
            "file_low": low_by_rep[rep].name,
            "file_high": high_by_rep[rep].name,
            "time_step": t1,
            "gene_corr": gene_corr,
            "twin_corr_gene1_gene2": twin_corr,
            "random_corr_gene1_gene2": random_corr
        })

# Collect into DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["file_low", "time_step"]).reset_index(drop=True)

# # Save table (optional)
results_df.to_csv(f"{path_to_plot_data}/twin_correlation_over_time_binomial_partition_{folder_name}.csv", index=False)


# Directional correlation

In [53]:
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm
from scipy.stats import spearmanr
import numpy as np
import matplotlib.patches as patches


def fast_corr(x, y):
    """Compute Spearman correlation between two 1D arrays."""
    corr, _ = spearmanr(x, y)
    return corr

def shuffle_test(x, y, threshold=0.01, n_shuffles=5000, rng=None):
    """Return True if Spearman correlation passes shuffle test."""
    actual_corr = fast_corr(x, y)

    # Create shuffled versions of y
    shuffles = np.array([rng.permutation(y) for _ in range(n_shuffles)])
    
    # Compute Spearman correlation for each shuffle
    shuffled_corrs = np.array([fast_corr(x, yy) for yy in shuffles])

    # Determine significance cutoff
    from scipy.stats import percentileofscore

    percentile_score = percentileofscore(np.abs(shuffled_corrs), abs(actual_corr))
    print(percentile_score)  # e.g. 97.3 means it's higher than 97.3% of values

    return percentile_score, actual_corr


# ------------------------------------------------
# Per time-pair function
# ------------------------------------------------
def process_time_pair(t1, t2, across_t_twin1, across_t_twin2,
                      csv_name, rng, all_gene_pairs, gene_pair,
                      threshold=0.01, n_shuffles=5000):
    if across_t_twin1.empty or across_t_twin2.empty:
        return None
        

    # Compute raw + normalized matrices for all gene pairs
    direction_raw_matrix, direction_normalized_matrix = get_cross_correlations(
        across_t_twin1, across_t_twin2, gene_pairs=all_gene_pairs
    )
    g1, g2 = gene_pair[0]
    if g1 not in direction_raw_matrix.index or g2 not in direction_raw_matrix.columns:
        return None

    raw_val = direction_raw_matrix.loc[g1, g2]
    norm_val = direction_normalized_matrix.loc[g1, g2]

    # Shuffle test on raw values
    x = across_t_twin1[f"{g1}_mRNA"].values
    y = across_t_twin2[f"{g2}_mRNA"].values
    percentile_score,_ = shuffle_test(x, y, threshold=threshold, n_shuffles=n_shuffles, rng=rng)

    

    return {
        "file": csv_name,
        "t1": t1,
        "t2": t2,
        "raw_corr": raw_val,
        "normalized_corr": norm_val,
        "percentile_score": percentile_score
    }

In [ ]:
# ------------------------------------------------
# Main loop
# ------------------------------------------------
seed = 1010
rng = np.random.default_rng(seed)
p_value_threshold_cross_correlation = 0.01
n_shuffles = 10000
gene_pair = [("gene_1", "gene_2")]   # <-- the pair you care about

#Path to simulation in figure 2 - A to B
folder = Path("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/"
              "median_parameter_simulations/figure_2_simulations_6000_cells/A_to_B/")

results = []
for i, csv_path in enumerate(folder.glob("df_rows_*.csv")):
    if i > 0:   # just first file for now
        break

    simulation = pd.read_csv(csv_path)
    n_clones_simulation = simulation['clone_id'].nunique()

    # shuffle clones
    clone_ids_shuffled = rng.permutation(n_clones_simulation)
    n1 = n2 = n_clones_simulation // 4
    across_t_clones = clone_ids_shuffled[n1 + n2:]

    all_times = sorted(simulation['time_step'].unique())
    tasks = []
    for t1 in all_times:
        for t2 in all_times:
            if t1 >= t2:
                continue
            across_t_twin1 = simulation[
                (simulation['clone_id'].isin(across_t_clones)) &
                (simulation['time_step'] == t1) &
                (simulation['replicate'] == 1)
            ]
            across_t_twin2 = simulation[
                (simulation['clone_id'].isin(across_t_clones)) &
                (simulation['time_step'] == t2) &
                (simulation['replicate'] == 2)
            ]
            tasks.append((t1, t2, across_t_twin1, across_t_twin2, csv_path.name))
    
    with tqdm_joblib(tqdm(desc="Processing tasks", total=len(tasks))):
        parallel_results = Parallel(n_jobs=-1)(
            delayed(process_time_pair)(*task, rng, all_gene_pairs=gene_pair,  # supply your pairs list
                                    gene_pair=gene_pair,
                                    threshold=p_value_threshold_cross_correlation,
                                    n_shuffles=n_shuffles)
            for task in tasks
        )
    results.extend([res for res in parallel_results if res is not None])

# ------------------------------------------------
# Aggregate results
# ------------------------------------------------
results_df = pd.DataFrame(results)


results_df.to_csv(f"{path_to_plot_data}/directional_corr_t1_t2.csv")

In [ ]:
# ------------------------------------------------
# Main loop
# ------------------------------------------------
seed = 1010
rng = np.random.default_rng(seed)
p_value_threshold_cross_correlation = 0.01
n_shuffles = 10000
gene_pair = [("gene_1", "gene_2")]   # <-- the pair you care about

#Path to simulation in figure 3 - A rep B

folder = Path("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/"
              "median_parameter_simulations/figure_2_simulations_6000_cells/A_rep_B/")

results = []
for i, csv_path in enumerate(folder.glob("df_rows_*.csv")):
    if i > 0:   # just first file for now
        break

    simulation = pd.read_csv(csv_path)
    n_clones_simulation = simulation['clone_id'].nunique()

    # shuffle clones
    clone_ids_shuffled = rng.permutation(n_clones_simulation)
    n1 = n2 = n_clones_simulation // 4
    across_t_clones = clone_ids_shuffled[n1 + n2:]

    all_times = sorted(simulation['time_step'].unique())
    tasks = []
    for t1 in all_times:
        for t2 in all_times:
            if t1 >= t2:
                continue
            across_t_twin1 = simulation[
                (simulation['clone_id'].isin(across_t_clones)) &
                (simulation['time_step'] == t1) &
                (simulation['replicate'] == 1)
            ]
            across_t_twin2 = simulation[
                (simulation['clone_id'].isin(across_t_clones)) &
                (simulation['time_step'] == t2) &
                (simulation['replicate'] == 2)
            ]
            tasks.append((t1, t2, across_t_twin1, across_t_twin2, csv_path.name))
    
    with tqdm_joblib(tqdm(desc="Processing tasks", total=len(tasks))):
        parallel_results = Parallel(n_jobs=-1)(
            delayed(process_time_pair)(*task, rng, all_gene_pairs=gene_pair,  # supply your pairs list
                                    gene_pair=gene_pair,
                                    threshold=p_value_threshold_cross_correlation,
                                    n_shuffles=n_shuffles)
            for task in tasks
        )
    results.extend([res for res in parallel_results if res is not None])

# ------------------------------------------------
# Aggregate results
# ------------------------------------------------
results_df = pd.DataFrame(results)


results_df.to_csv(f"{path_to_plot_data}/directional_corr_t1_t2_A_rep_B.csv")

# Scrambled distributions

## Gene-gene correlations

In [2]:
def compute_correlation_matrix(gene_matrix_1, gene_matrix_2, gene_list, gene_pairs=None):
   """Compute Spearman correlations between gene expression matrices."""
   n_genes = len(gene_list)
   gene_to_idx = {gene: i for i, gene in enumerate(gene_list)}
   raw_matrix = np.zeros((n_genes, n_genes))
   
   # Determine which pairs to compute
   if gene_pairs is None:
       pairs_to_compute = [(i, j) for i in range(n_genes) for j in range(n_genes)]
   else:
       pairs_to_compute = []
       for gene_1, gene_2 in gene_pairs:
           if gene_1 in gene_to_idx and gene_2 in gene_to_idx:
               i, j = gene_to_idx[gene_1], gene_to_idx[gene_2]
               pairs_to_compute.append((i, j))
   
   # Compute correlations
   for i, j in pairs_to_compute:
       corr = spearmanr(gene_matrix_1[i, :], gene_matrix_2[j, :]).correlation
       raw_matrix[i, j] = corr if not np.isnan(corr) else 0.0
   
   return pd.DataFrame(raw_matrix, index=gene_list, columns=gene_list)

def single_cell_shuffle(gene_matrix_1, gene_matrix_2, gene_list, shuffle_pairs):
            n_cells = gene_matrix_1.shape[1]
            shuffled_indices = np.random.permutation(n_cells)
            return compute_correlation_matrix(gene_matrix_1, gene_matrix_2[:, shuffled_indices], 
                                                  gene_list, shuffle_pairs)

In [ ]:
from itertools import combinations

#Simulation from figure 2
csv_path = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_to_B/df_rows_0_1_05082025_160041_ncells_6000_A_to_B_rep_0.csv"
simulation = pd.read_csv(csv_path)
path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_to_B.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 101010
np.random.seed(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)

pairwise_gene_gene_correlation_matrix = calculate_pairwise_gene_gene_correlation_matrix(
        all_t1_t2_measurements, gene_list
    )

# Extract gene matrices from DataFrame
gene_matrix = []

for gene in gene_list:
    # Look for gene columns (adapt this to your column naming)
    gene_col = f"{gene}_mRNA"  # Adjust this pattern as needed
    if gene_col in all_t1_t2_measurements.columns:
        # Split by time point or condition - adjust this logic for your data structure
        gene_data = all_t1_t2_measurements[gene_col].values
        gene_matrix.append(gene_data)
    else:
        raise ValueError(f"Could not find column for gene: {gene}")

gene_matrix = np.array(gene_matrix)  # Shape: (n_genes, n_cells)
all_pairs = list(combinations(gene_list, 2))  # Unique undirected pairs

pair_correlations = {(gi, gj): pairwise_gene_gene_correlation_matrix.loc[gi, gj] for gi, gj in all_pairs}
      
    # Generate null distribution
    
n_cores_to_use = max(1, os.cpu_count() - 2)
shuffled_results = Parallel(n_jobs=n_cores_to_use, verbose=0)(
        delayed(single_cell_shuffle)(gene_matrix, gene_matrix, gene_list, all_pairs) 
        for _ in range(n_shuffles)
)
    
percentile_threshold = (1 - p_val_threshold) * 100

no_regulation, potential_regulation = [], []

outname = f"gene_gene_corr_A_to_B" 

gi, gj = all_pairs [0]
corr_val = pair_correlations[(gi, gj)]
shuffled_vals = [r.loc[gi, gj] for r in shuffled_results]
current_threshold = np.percentile(shuffled_vals, percentile_threshold)
p_value = np.mean(np.array(shuffled_vals) >= abs(corr_val))
direction_str = "-"


df_out = pd.DataFrame({
        "pair": f"{gi}_{gj}",
        "shuffled_vals": shuffled_vals
    })
df_out["actual_corr"] = corr_val
df_out["threshold"] = current_threshold
df_out["p_value"] = p_value

df_out.to_csv(f"{path_to_plot_data}/{outname}.csv", index=False)

In [ ]:
from itertools import combinations
#Simulation from figure 2 - A_B type
csv_path = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_B/df_rows_0_1_08082025_093223_ncells_6000_A_B_rep_0.csv"
simulation = pd.read_csv(csv_path)
path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_B.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 101010
np.random.seed(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)

pairwise_gene_gene_correlation_matrix = calculate_pairwise_gene_gene_correlation_matrix(
        all_t1_t2_measurements, gene_list
    )

# Extract gene matrices from DataFrame
gene_matrix = []

for gene in gene_list:
    # Look for gene columns (adapt this to your column naming)
    gene_col = f"{gene}_mRNA"  # Adjust this pattern as needed
    if gene_col in all_t1_t2_measurements.columns:
        # Split by time point or condition - adjust this logic for your data structure
        gene_data = all_t1_t2_measurements[gene_col].values
        gene_matrix.append(gene_data)
    else:
        raise ValueError(f"Could not find column for gene: {gene}")

gene_matrix = np.array(gene_matrix)  # Shape: (n_genes, n_cells)
all_pairs = list(combinations(gene_list, 2))  # Unique undirected pairs

pair_correlations = {(gi, gj): pairwise_gene_gene_correlation_matrix.loc[gi, gj] for gi, gj in all_pairs}
      
    # Generate null distribution
    
n_cores_to_use = max(1, os.cpu_count() - 2)
shuffled_results = Parallel(n_jobs=n_cores_to_use, verbose=0)(
        delayed(single_cell_shuffle)(gene_matrix, gene_matrix, gene_list, all_pairs) 
        for _ in range(n_shuffles)
)
    
percentile_threshold = (1 - p_val_threshold) * 100

no_regulation, potential_regulation = [], []

outname = f"gene_gene_corr_A_B"

gi, gj = all_pairs [0]
corr_val = pair_correlations[(gi, gj)]
shuffled_vals = [r.loc[gi, gj] for r in shuffled_results]
current_threshold = np.percentile(shuffled_vals, percentile_threshold)
p_value = np.mean(np.array(shuffled_vals) >= abs(corr_val))
direction_str = "-"


df_out = pd.DataFrame({
        "pair": f"{gi}_{gj}",
        "shuffled_vals": shuffled_vals
    })
df_out["actual_corr"] = corr_val
df_out["threshold"] = current_threshold
df_out["p_value"] = p_value

df_out.to_csv(f"{path_to_plot_data}/{outname}.csv", index=False)

In [ ]:
# ####CURRENT###########
from itertools import combinations
#Path to simulation files in figure 2 - the high and low of A_B type
csv_path_low = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_B_low_k_on/df_rows_2_2_09082025_210256_ncells_6000_A_B_low_k_on_rep_0.csv"
csv_path_high = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_B_high_k_on/df_rows_3_3_09082025_201359_ncells_6000_A_B_high_k_on_rep_0.csv"

simulation = split_and_merge_simulations([csv_path_low, csv_path_high])
path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_B.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 101010
np.random.seed(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)

pairwise_gene_gene_correlation_matrix = calculate_pairwise_gene_gene_correlation_matrix(
        all_t1_t2_measurements, gene_list
    )

# Extract gene matrices from DataFrame
gene_matrix = []

for gene in gene_list:
    # Look for gene columns (adapt this to your column naming)
    gene_col = f"{gene}_mRNA"  # Adjust this pattern as needed
    if gene_col in all_t1_t2_measurements.columns:
        # Split by time point or condition - adjust this logic for your data structure
        gene_data = all_t1_t2_measurements[gene_col].values
        gene_matrix.append(gene_data)
    else:
        raise ValueError(f"Could not find column for gene: {gene}")

gene_matrix = np.array(gene_matrix)  # Shape: (n_genes, n_cells)
all_pairs = list(combinations(gene_list, 2))  # Unique undirected pairs

pair_correlations = {(gi, gj): pairwise_gene_gene_correlation_matrix.loc[gi, gj] for gi, gj in all_pairs}
      
    # Generate null distribution
    
n_cores_to_use = max(1, os.cpu_count() - 2)
shuffled_results = Parallel(n_jobs=n_cores_to_use, verbose=0)(
        delayed(single_cell_shuffle)(gene_matrix, gene_matrix, gene_list, all_pairs) 
        for _ in range(n_shuffles)
)
    
percentile_threshold = (1 - p_val_threshold) * 100
outname = f"gene_gene_corr_A_B_multistate"
gi, gj = all_pairs [0]
corr_val = pair_correlations[(gi, gj)]
shuffled_vals = [r.loc[gi, gj] for r in shuffled_results]
current_threshold = np.percentile(shuffled_vals, percentile_threshold)
p_value = np.mean(np.array(shuffled_vals) >= abs(corr_val))
direction_str = "-"


df_out = pd.DataFrame({
        "pair": f"{gi}_{gj}",
        "shuffled_vals": shuffled_vals
    })
df_out["actual_corr"] = corr_val
df_out["threshold"] = current_threshold
df_out["p_value"] = p_value

df_out.to_csv(f"{path_to_plot_data}/{outname}.csv", index=False)

In [ ]:
from itertools import combinations
#Path to simulation files in figure 2 - the high and low of A_to_B type

csv_path_low = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_to_B_low_k_on/df_rows_2_2_09082025_212311_ncells_6000_A_to_B_low_k_on_rep_0.csv"
csv_path_high = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_to_B_high_k_on/df_rows_3_3_09082025_220257_ncells_6000_A_to_B_high_k_on_rep_0.csv"

simulation = split_and_merge_simulations([csv_path_low, csv_path_high])
path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_to_B.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 101010
np.random.seed(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)

pairwise_gene_gene_correlation_matrix = calculate_pairwise_gene_gene_correlation_matrix(
        all_t1_t2_measurements, gene_list
    )

# Extract gene matrices from DataFrame
gene_matrix = []

for gene in gene_list:
    # Look for gene columns (adapt this to your column naming)
    gene_col = f"{gene}_mRNA"  # Adjust this pattern as needed
    if gene_col in all_t1_t2_measurements.columns:
        # Split by time point or condition - adjust this logic for your data structure
        gene_data = all_t1_t2_measurements[gene_col].values
        gene_matrix.append(gene_data)
    else:
        raise ValueError(f"Could not find column for gene: {gene}")

gene_matrix = np.array(gene_matrix)  # Shape: (n_genes, n_cells)
all_pairs = list(combinations(gene_list, 2))  # Unique undirected pairs

pair_correlations = {(gi, gj): pairwise_gene_gene_correlation_matrix.loc[gi, gj] for gi, gj in all_pairs}
      
    # Generate null distribution
    
n_cores_to_use = max(1, os.cpu_count() - 2)
shuffled_results = Parallel(n_jobs=n_cores_to_use, verbose=0)(
        delayed(single_cell_shuffle)(gene_matrix, gene_matrix, gene_list, all_pairs) 
        for _ in range(n_shuffles)
)
    
percentile_threshold = (1 - p_val_threshold) * 100

no_regulation, potential_regulation = [], []
outname = f"gene_gene_corr_A_to_B_multistate"
gi, gj = all_pairs [0]
corr_val = pair_correlations[(gi, gj)]
shuffled_vals = [r.loc[gi, gj] for r in shuffled_results]
current_threshold = np.percentile(shuffled_vals, percentile_threshold)
p_value = np.mean(np.array(shuffled_vals) >= abs(corr_val))
direction_str = "-"


df_out = pd.DataFrame({
        "pair": f"{gi}_{gj}",
        "shuffled_vals": shuffled_vals
    })
df_out["actual_corr"] = corr_val
df_out["threshold"] = current_threshold
df_out["p_value"] = p_value

df_out.to_csv(f"{path_to_plot_data}/{outname}.csv", index=False)

## Twin vs random

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, linregress, pearsonr
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from itertools import combinations, permutations
import os
from joblib import Parallel, delayed

def generate_random_shuffle(simulation_data, gene_list, n_shuffles=10000, random_state=42):
   np.random.seed(random_state)
   
   rep_0 = simulation_data[simulation_data['replicate'] == 1].reset_index(drop=True)
   rep_1 = simulation_data[simulation_data['replicate'] == 2].reset_index(drop=True)
   gene_cols = [f"{gene}_mRNA" for gene in gene_list]
   min_cells = min(len(rep_0), len(rep_1))
   expr_0 = rep_0[gene_cols].iloc[:min_cells].values
   expr_1 = rep_1[gene_cols].iloc[:min_cells].values

   n_cells, n_genes = expr_0.shape
   triu_indices = np.triu_indices(n_genes, k=1)
   gene_pairs = [(gene_list[i], gene_list[j]) for i, j in zip(triu_indices[0], triu_indices[1])]
   
   all_shuffle_indices = np.array([np.random.permutation(n_cells) for _ in range(n_shuffles)])
   all_correlations = np.zeros((n_shuffles, len(gene_pairs)))
   
   for batch_start in range(0, n_shuffles, 100):
       batch_end = min(batch_start + 100, n_shuffles)
       for i, shuffle_idx in enumerate(all_shuffle_indices[batch_start:batch_end]):
           expr_1_shuffled = expr_1[shuffle_idx]
           deltas = expr_0 - expr_1_shuffled
           ranked_deltas = np.apply_along_axis(rankdata, 0, deltas)
           corr_matrix = np.corrcoef(ranked_deltas.T)
           all_correlations[batch_start + i] = corr_matrix[triu_indices]
   
   # Store with sorted keys to avoid duplicates
   correlation_dict = {}
   for i, (gene_i, gene_j) in enumerate(gene_pairs):
       key = tuple(sorted([gene_i, gene_j]))
       correlation_dict[key] = all_correlations[:, i]
   
   return correlation_dict

def get_correlations(correlation_dict, gene_i, gene_j):
   return correlation_dict[tuple(sorted([gene_i, gene_j]))]

from itertools import combinations
#Path to simulation files in figure 2 A_to_B type

csv_path = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_to_B/df_rows_0_1_05082025_160041_ncells_6000_A_to_B_rep_0.csv"
simulation = pd.read_csv(csv_path)
path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_to_B.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 101010
np.random.seed(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)
twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t2 = calculate_twin_random_pair_correlations(
        all_t1_t2_measurements, t1_twins, gene_list
    )
random_pair_correlation_distribution = generate_random_shuffle(all_t1_t2_measurements, gene_list=gene_list)
all_pairs = list(combinations(gene_list, 2))  # Unique undirected pairs
gene_i, gene_j = all_pairs[0]
t_corr = twin_pair_correlation_matrix_t1.loc[gene_i, gene_j]
r_corr = get_correlations(random_pair_correlation_distribution, gene_i, gene_j)
r_corr_std = np.std(r_corr)

z_score = (t_corr - np.mean(r_corr)) / r_corr_std
threshold_corr = -10*r_corr_std + np.mean(r_corr)
outname = f"twin_vs_random_corr_A_to_B"
df_out = pd.DataFrame({"random_corr": r_corr})
df_out["twin_corr"] = t_corr
df_out["z_score"] = z_score
df_out["threshold_corr"] = threshold_corr
df_out.to_csv(
        f"{path_to_plot_data}/{outname}.csv",
        index=False
    )


In [ ]:
def split_and_merge_simulations(path_to_simulation_file):
    """Split clone IDs evenly between two simulations and merge them."""
    
    simulation_1 = pd.read_csv(path_to_simulation_file[0])
    simulation_2 = pd.read_csv(path_to_simulation_file[1])
    
    # Get unique clone IDs and split in half
    clone_ids = sorted(simulation_1['clone_id'].unique())
    half_point = len(clone_ids) // 2
    
    clones_from_sim1 = clone_ids[:half_point]
    clones_from_sim2 = clone_ids[half_point:]
    
    # Filter and merge
    sim1_subset = simulation_1[simulation_1['clone_id'].isin(clones_from_sim1)]
    sim2_subset = simulation_2[simulation_2['clone_id'].isin(clones_from_sim2)]
    merged_df = pd.concat([sim1_subset, sim2_subset], ignore_index=True)
    return merged_df

In [ ]:
#Simulations from figure 2 - the high and low of the A_B type
csv_path_1  = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_B_high_k_on/df_rows_3_3_09082025_201359_ncells_6000_A_B_high_k_on_rep_0.csv"
csv_path_2  = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_B_low_k_on/df_rows_2_2_09082025_210256_ncells_6000_A_B_low_k_on_rep_0.csv"
path_to_simulation_file = [csv_path_1, csv_path_2]
simulation = split_and_merge_simulations(path_to_simulation_file)
path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_B.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 101010
np.random.seed(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)
twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t2 = calculate_twin_random_pair_correlations(
        all_t1_t2_measurements, t1_twins, gene_list
    )
random_pair_correlation_distribution = generate_random_shuffle(all_t1_t2_measurements, gene_list=gene_list)
all_pairs = list(combinations(gene_list, 2))  # Unique undirected pairs
gene_i, gene_j = all_pairs[0]
t_corr = twin_pair_correlation_matrix_t1.loc[gene_i, gene_j]
r_corr = get_correlations(random_pair_correlation_distribution, gene_i, gene_j)
r_corr_std = np.std(r_corr)

z_score = (t_corr - np.mean(r_corr)) / r_corr_std
threshold_corr = -10*r_corr_std + np.mean(r_corr)
outname = f"twin_vs_random_corr_A_B_2_states"
df_out = pd.DataFrame({"random_corr": r_corr})
df_out["twin_corr"] = t_corr
df_out["z_score"] = z_score
df_out["threshold_corr"] = threshold_corr
df_out.to_csv(
        f"{path_to_plot_data}/{outname}.csv",
        index=False
    )

In [ ]:
#Simulations from figure 2 - the high and low of the A_to_B type

csv_path_1  = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_to_B_high_k_on/df_rows_3_3_09082025_220257_ncells_6000_A_to_B_high_k_on_rep_0.csv"
csv_path_2  = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_to_B_low_k_on/df_rows_2_2_09082025_212311_ncells_6000_A_to_B_low_k_on_rep_0.csv"
path_to_simulation_file = [csv_path_1, csv_path_2]
simulation = split_and_merge_simulations(path_to_simulation_file)
path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_to_B.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 101010
np.random.seed(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)
twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t2 = calculate_twin_random_pair_correlations(
        all_t1_t2_measurements, t1_twins, gene_list
    )
random_pair_correlation_distribution = generate_random_shuffle(all_t1_t2_measurements, gene_list=gene_list)
all_pairs = list(combinations(gene_list, 2))  # Unique undirected pairs
gene_i, gene_j = all_pairs[0]
t_corr = twin_pair_correlation_matrix_t1.loc[gene_i, gene_j]
r_corr = get_correlations(random_pair_correlation_distribution, gene_i, gene_j)
r_corr_std = np.std(r_corr)

z_score = (t_corr - np.mean(r_corr)) / r_corr_std
threshold_corr = -10*r_corr_std + np.mean(r_corr)
outname = f"twin_vs_random_corr_A_to_B_2_states"
df_out = pd.DataFrame({"random_corr": r_corr})
df_out["twin_corr"] = t_corr
df_out["z_score"] = z_score
df_out["threshold_corr"] = threshold_corr
df_out.to_csv(
        f"{path_to_plot_data}/{outname}.csv",
        index=False
    )

## Scrambled control for directional correlations

In [ ]:
#SImulation from figure 2 - A_to_B
path_to_simulation_file = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/A_to_B/df_rows_0_1_05082025_160041_ncells_6000_A_to_B_rep_0.csv"
simulation = pd.read_csv(path_to_simulation_file)
path_to_connectivity_matrix = f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_to_B.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 101010
rng = np.random.default_rng(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)
twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t2 = calculate_twin_random_pair_correlations(
        all_t1_t2_measurements, t1_twins, gene_list
    )
random_pair_correlation_distribution = generate_random_shuffle(all_t1_t2_measurements, gene_list=gene_list)
all_pairs = list(combinations(gene_list, 2))  # Unique undirected pairs
import matplotlib as mpl
import pandas as pd
mpl.rcParams['svg.fonttype'] = 'none'   # keep text editable in SVG

direction_raw_matrix, direction_normalized_matrix = get_cross_correlations(
        across_t_twin1, across_t_twin2, gene_pairs=all_pairs
    )
g1, g2 = all_pairs[0]

raw_val = direction_raw_matrix.loc[g1, g2]
norm_val = direction_normalized_matrix.loc[g1, g2]
# Shuffle test on raw values
n_shuffles = 10000
x = across_t_twin1[f"{g1}_mRNA"].values
y = across_t_twin2[f"{g2}_mRNA"].values
shuffles = np.array([rng.permutation(y) for _ in range(n_shuffles)])
    
# Compute Spearman correlation for each shuffle
shuffled_corrs = np.array([fast_corr(x, yy) for yy in shuffles])
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- Settings ---
percentile_threshold = 99
hist_color = "#d9d9d9"  # fixed histogram color
line_colors = {
    "blue":  "#2171b5",
    "red":   "#cb181d",
    "grey":  "#525252",
}
choice = "blue"  # or "red", "grey"

# Compute threshold from shuffled distribution
current_threshold = np.percentile(np.abs(shuffled_corrs), percentile_threshold)

# Empirical p-value
p_value = np.mean(np.abs(shuffled_corrs) >= abs(raw_val))
# --- Save CSV with shuffle + actual values ---
outname = f"direction_A_to_B"
df_out = pd.DataFrame({"shuffled_corrs": shuffled_corrs})
df_out["actual_corr"] = raw_val
df_out["normalized_corr"] = norm_val
df_out["threshold"] = current_threshold
df_out["p_value"] = p_value
df_out.to_csv(
    f"{path_to_plot_data}/{outname}.csv",
    index=False
)

In [ ]:
path_to_simulation_file = "/home/keerthm/twinfer/df_rows_0_1_05082025_160158_ncells_6000_A_to_B_rep_1.csv"
simulation = pd.read_csv(path_to_simulation_file)
path_to_connectivity_matrix = f"/home/keerthm/twinfer/matrix.txt"
n_clones_simulation = simulation['clone_id'].nunique()
n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

p_val_threshold = 0.01
n_shuffles=10000

# Shuffle clones
seed = 10110
rng = np.random.default_rng(seed)
clone_ids_shuffled = np.random.permutation(n_clones_simulation)
n1 = n2 = n_clones_simulation // 4
t1_clones = clone_ids_shuffled[:n1]
t2_clones = clone_ids_shuffled[n1:n1 + n2]
across_t_clones = clone_ids_shuffled[n1 + n2:]

t1 = 1
t2 = 20
t1_twins = simulation[simulation['clone_id'].isin(t1_clones) & (simulation['time_step'] == t1)]

t2_twins = simulation[simulation['clone_id'].isin(t2_clones) & (simulation['time_step'] == t2)]
# Across_t: pick exactly one random twin per clone_id
# One cell per clone at t1
across_t_twin1 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t1) & (simulation['replicate'] == 1)]
)

across_t_twin2 = (
    simulation[(simulation['clone_id'].isin(across_t_clones)) & (simulation['time_step'] == t2) & (simulation['replicate'] == 2)]
)
# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)
across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)
all_t1_t2_measurements = pd.concat(
[t1_twins, t2_twins, across_t_twin1, across_t_twin2],
ignore_index=True
)
twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t2 = calculate_twin_random_pair_correlations(
        all_t1_t2_measurements, t1_twins, gene_list
    )
random_pair_correlation_distribution = generate_random_shuffle(all_t1_t2_measurements, gene_list=gene_list)
all_pairs = list(permutations(gene_list, 2))  # Unique undirected pairs
import matplotlib as mpl
import pandas as pd
mpl.rcParams['svg.fonttype'] = 'none'   # keep text editable in SVG

direction_raw_matrix, direction_normalized_matrix = get_cross_correlations(
        across_t_twin1, across_t_twin2, gene_pairs=all_pairs
    )
g1, g2 = all_pairs[0]

raw_val = direction_raw_matrix.loc[g2, g1]
norm_val = direction_normalized_matrix.loc[g2, g1]
# Shuffle test on raw values
n_shuffles = 10000
x = across_t_twin2[f"{g1}_mRNA"].values
y = across_t_twin1[f"{g2}_mRNA"].values
shuffles = np.array([rng.permutation(y) for _ in range(n_shuffles)])
    
# Compute Spearman correlation for each shuffle
shuffled_corrs = np.array([fast_corr(x, yy) for yy in shuffles])
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- Settings ---
percentile_threshold = 99
hist_color = "#d9d9d9"  # fixed histogram color
line_colors = {
    "blue":  "#2171b5",
    "red":   "#cb181d",
    "grey":  "#525252",
}
choice = "grey"  # or "red", "grey"

# Compute threshold from shuffled distribution
current_threshold = np.percentile(np.abs(shuffled_corrs), percentile_threshold)

# Empirical p-value
p_value = np.mean(np.abs(shuffled_corrs) >= abs(raw_val))

# --- Save CSV with shuffle + actual values ---
df_out = pd.DataFrame({"shuffled_corrs": shuffled_corrs})
outname = f"direction_B_to_A"
df_out["actual_corr"] = raw_val
df_out["normalized_corr"] = norm_val
df_out["threshold"] = current_threshold
df_out["p_value"] = p_value
df_out.to_csv(
    f"{path_to_plot_data}/{outname}.csv",
    index=False
)